# PDF to RDA DMP JSON

One simple prompt. Give `llama3.1:8b` the text of a DMP PDF and the complete
**maDMP 1.2** schema, and ask for JSON that complies with the schema, extracting
whatever information the DMP contains.

| Step | What happens |
|---|---|
| 1 | Read sample 14 with pdfplumber |
| 2 | Load the schema |
| 3 | Prompt: schema + DMP text → JSON |
| 4 | Save |


In [1]:
import json
from pathlib import Path

if Path.cwd().name == "notebooks":
    import os
    os.chdir(Path.cwd().parent)

from dmpbridge.extractors import get_extractor
from dmpbridge.models.ollama import OllamaModel

PDF    = Path("data/input/pdfs/sample14.pdf")
SCHEMA = Path("data/output/rda/maDMP-schema-1.2.json")
MODEL  = "llama3.1:8b"
OUT    = Path("data/output/rda") / (PDF.stem + ".rda.json")


## Step 1 — Read the PDF


In [2]:
dmp_text = get_extractor("pdfplumber").extract(PDF)[0]["text"]

print(f"{len(dmp_text):,} characters\n")
print(dmp_text[:500])


12,930 characters

** Plan Overview **
_ A Data Management Plan created using DMPTool _
** DMP ID: ** https://doi.org/10.48321/D1CW23
** Title: ** Hakai Institute Juvenile Salmon Program Time Series
** Creator: ** Brett Johnson - ** ORCID: ** ++ 0000-0001-9317-0364 ++
** Affiliation: ** Hakai Institute
** Principal Investigator: ** Brett Johnson, Brian Hunt
** Data Manager: ** Brett Johnson, Tim van der Stap, Krystal Bachen
** Funder: ** Tula Foundation
** Template: ** Hakai Institute Data Management Plan
** Proje


## Step 2 — Load the schema

The schema uses `$ref` pointers into its `$defs` section. They are replaced with
the definitions they point to, so the model sees the complete schema in one piece.


In [3]:
schema = json.loads(SCHEMA.read_text(encoding="utf-8"))
defs = schema["$defs"]


def inline_refs(node):
    """Replace every $ref with the definition it points at."""
    if isinstance(node, dict):
        if "$ref" in node:
            return inline_refs(defs[node["$ref"].split("/")[-1]])
        return {k: inline_refs(v) for k, v in node.items() if k != "$defs"}
    if isinstance(node, list):
        return [inline_refs(v) for v in node]
    return node


def without(node, keys, in_properties=False):
    """Drop the given annotation keys — but never a field NAME: dmp.title,
    dataset.description and distribution.format are real fields that happen
    to share a name with schema annotations."""
    if isinstance(node, dict):
        out = {}
        for k, v in node.items():
            if in_properties:
                out[k] = without(v, keys)
            elif k == "properties":
                out[k] = without(v, keys, in_properties=True)
            elif k not in keys:
                out[k] = without(v, keys)
        return out
    if isinstance(node, list):
        return [without(v, keys) for v in node]
    return node


full = inline_refs(schema)

# For the prompt: the complete schema, minus its "examples" — the model copied
# example values (a sample funder ID, a sample grant URL) into the output as
# if the DMP had stated them.
schema_text = json.dumps(without(full, {"examples"}), separators=(",", ":"))

# For Ollama's `format`: the structure only. Ollama constrains decoding to it,
# which is what makes the output follow the schema — and stop.
format_schema = without(full, {"description", "title", "examples", "format", "$schema", "$id"})

print(f"{len(schema_text):,} characters of schema in the prompt")
print(f"{len(json.dumps(format_schema)):,} characters of schema as the output constraint")


31,097 characters of schema in the prompt
15,693 characters of schema as the output constraint


## Step 3 — Prompt

The schema is given to the model twice: as text in the prompt, and as Ollama's
`format`, which constrains the JSON it generates to that structure. Without the
constraint, `llama3.1:8b` fell into a loop on this prompt — the same contributor
block repeated 47 times — and never stopped. `num_predict` is a cap in case it
ever does again.

One consequence to know about: the schema **requires** `contact.mbox`, `created`
and `modified`. This DMP states none of them, so the model fills them in itself.


In [4]:
llm = OllamaModel(model=MODEL, host="http://localhost:11434",
                  num_ctx=32768, num_predict=8000)

PROMPT = f"""Here is the RDA maDMP JSON schema (version 1.2):

{schema_text}

Here is the text of a Data Management Plan:

{dmp_text}

Generate a JSON document that complies with the schema above, extracting whatever
information is possible from the Data Management Plan text. Output only the JSON."""

raw = llm.complete("You convert Data Management Plans into RDA maDMP JSON.",
                   PROMPT, schema=format_schema)
result = json.loads(raw)

print(json.dumps(result, indent=2, ensure_ascii=False))


{
  "dmp": {
    "contact": {
      "contact_id": [
        {
          "identifier": "0000-0001-9317-0364",
          "type": "orcid"
        }
      ],
      "mbox": "brett.johnson@hakai.org",
      "name": "Brett Johnson"
    },
    "created": "2022-01-01T12:00:00Z",
    "dataset": [
      {
        "dataset_id": {
          "identifier": "https://doi.org/10.48321/D1CW23",
          "type": "doi"
        },
        "personal_data": "no",
        "sensitive_data": "no",
        "title": "Hakai Institute Juvenile Salmon Program Time Series",
        "type": "dataset"
      }
    ],
    "dmp_id": {
      "identifier": "https://doi.org/10.48321/D1CW23",
      "type": "doi"
    },
    "ethical_issues_exist": "no",
    "language": "eng",
    "modified": "2024-06-11T12:00:00Z",
    "title": "Hakai Institute Juvenile Salmon Program Time Series Data Management Plan"
  }
}


## Step 4 — Save


In [5]:
OUT.parent.mkdir(parents=True, exist_ok=True)
OUT.write_text(json.dumps(result, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"saved -> {OUT}")


saved -> data\output\rda\sample14.rda.json
